# Mission 2 로컬 실행 노트북

Colab 대신 로컬 PC에서 돌리기 위한 노트북입니다. 이 파이프라인(음향 통계 feature + LightGBM)은 GPU를 전혀 쓰지 않으므로 CPU/디스크만 있으면 됩니다.

**시작하기 전에**: 우측 상단에서 커널을 이 저장소의 `.venv` (`데이터 크리에이터 캠프\.venv\Scripts\python.exe`)로 선택하세요. `ipykernel`/`librosa`/`pandas`/`lightgbm`이 이미 다 설치되어 있습니다.

In [ ]:
import os

REPO_ROOT = r"C:\Users\hhjkk\OneDrive\바탕 화면\데이터 크리에이터 캠프\prelim"
os.chdir(REPO_ROOT)
print("cwd:", os.getcwd())

In [ ]:
import subprocess
import sys

# Hardcoded rather than sys.executable: VS Code has reset the notebook's
# kernel back to a different (non-.venv) Python a couple of times already
# after reopening this file, and that Python doesn't have librosa/lightgbm
# installed -- pinning this here means run_step() always uses the right
# interpreter for scripts/*.py regardless of whatever kernel got selected.
VENV_PYTHON = r"C:\Users\hhjkk\OneDrive\바탕 화면\데이터 크리에이터 캠프\.venv\Scripts\python.exe"


def run_step(args: list[str]) -> None:
    """Run a scripts/*.py CLI, streaming its output live (so a long step like
    the feature build still shows progress) while guaranteeing the real
    error text is always visible if it fails -- plain
    subprocess.run(..., check=True) only raises CalledProcessError, whose
    repr does not include the script's own printed diagnostics, which made a
    couple of real failures here hard to debug from just the traceback."""
    process = subprocess.Popen(
        [VENV_PYTHON, *args],
        cwd=REPO_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        env={**os.environ, "PYTHONIOENCODING": "utf-8"},
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f"실패 (exit {process.returncode}): {' '.join(args)}")

## 0. 경로 설정 -- ZIP_DIR을 실제 경로로 바꾸세요

예전에 Colab에서 이 자리를 placeholder 그대로 돌려서 조용히 일부만 처리된 적이 있었습니다. 그래서 아래 assert가 실제로 zip 파일을 찾았는지 먼저 검증합니다 -- 여기서 에러가 나면 다음 단계로 넘어가지 마세요.

In [ ]:
from pathlib import Path

ZIP_DIR = r"C:\Users\hhjkk\OneDrive\바탕 화면\데이터 크리에이터 캠프\대학부 데이터"
OUTPUT_DIR = r"C:\dcc_data_local_full"  # C드라이브 여유 36.79GB 중 ~34GB 사용 예정 -- 여유가 빠듯함, 아래 디스크 경고 참고
CACHE_DIR = Path(REPO_ROOT) / "data" / "local_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# 물리 12코어지만 이 PC 총 RAM이 15.6GB뿐이고(확인 시점 여유 RAM 2.67GB),
# 워커마다 BLAS 스레드 폭증으로 --workers 11에서 전부 크래시난 적 있음
# (지금은 mission2_features.py에서 BLAS 스레드를 1로 고정해서 그 문제 자체는
# 고쳤지만, 그래도 여유를 두는 게 안전함).
WORKERS = 8

zip_dir = Path(ZIP_DIR)
zips = list(zip_dir.rglob("*.zip"))  # 실제 zip이 한 겹 안쪽 폴더에 들어있어서 rglob 사용
assert zip_dir.is_dir(), f"경로가 존재하지 않습니다: {zip_dir}"
assert len(zips) > 0, f"zip 파일을 못 찾았습니다: {zip_dir}"
print(f"zip {len(zips)}개 확인됨 (재귀 탐색): {zip_dir}")
print(f"추출 대상 폴더: {OUTPUT_DIR}")
print(f"결과 캐시 폴더: {CACHE_DIR}")
print(f"feature 빌드 워커 수: {WORKERS}")

In [ ]:
import shutil as _shutil

_free_gb = _shutil.disk_usage("C:\\").free / 1e9
_source_gb = sum(f.stat().st_size for f in zip_dir.rglob("*") if f.is_file()) / 1e9
print(f"C드라이브 여유: {_free_gb:.1f} GB")
print(f"원본(zip+이미 풀린 것) 총 용량: {_source_gb:.1f} GB -- 압축 해제 후 OUTPUT_DIR도 비슷한 규모가 됩니다")
if _free_gb < _source_gb * 1.1:
    print("[경고] 여유 공간이 빠듯합니다. 도중에 꽉 차서 실패해도 --resume으로 이어할 수는 있지만,")
    print("       가능하면 미리 다른 파일을 정리해서 여유를 좀 더 확보하는 걸 추천합니다.")

## 1. zip 추출 (전체 29,200 + Validation 50) -- 재실행해도 이미 받은 건 건너뜁니다

In [ ]:
run_step(
    [
        "scripts/extract_dcc_sample_from_zips.py",
        "--zip-dir", ZIP_DIR,
        "--output-dir", OUTPUT_DIR,
        "--n-train-calls", "0",
        "--n-val-calls", "50",
        "--seed", "42",
    ]
)

출력의 "신규 N개 / 건너뜀 M개 / 전체 K개"에서 **N+M == 29,250**인지 꼭 확인하세요.

## 1b. 이미 통째로 풀려있던 chunk 병합

`대학부 데이터` 폴더 안에 zip이 아니라 이미 통째로 풀려있는 chunk가 하나 있었습니다(예전에 C드라이브에서 직접 풀다가 남은 것으로 추정). zip 스캔으로는 안 잡히니 따로 복사해서 합칩니다.

In [ ]:
import shutil


def _find_data_root(chunk: Path) -> Path | None:
    """The already-extracted chunk has Training/Validation nested one level
    under its own "대학부 데이터" subfolder -- find whichever level actually
    holds them, generically (not hardcoded to a specific chunk name)."""
    for name in ("Training", "Validation"):
        for hit in chunk.rglob(name):
            return hit.parent
    return None


already_extracted_chunks = [
    chunk
    for chunk in sorted(p for p in zip_dir.iterdir() if p.is_dir())
    if not list(chunk.rglob("*.zip"))  # has no zip inside -> not handled by step 1
]

print(f"zip 없이 이미 풀려있는 chunk: {len(already_extracted_chunks)}개")
for chunk in already_extracted_chunks:
    data_root = _find_data_root(chunk)
    if data_root is None:
        print(f"  [주의] {chunk} 안에서 Training/Validation을 못 찾음 -- 수동 확인 필요")
        continue
    shutil.copytree(data_root, OUTPUT_DIR, dirs_exist_ok=True)
    print(f"  병합 완료: {data_root} -> {OUTPUT_DIR}")

## 2. manifest 생성

In [ ]:
run_step(
    [
        "scripts/build_manifest.py",
        "--data-root", OUTPUT_DIR,
        "--output", "data/manifests/dcc_manifest_local_full.csv",
        "--overwrite",  # 추출을 재실행해서 파일이 늘었으면 manifest도 다시 만들어야 함
    ]
)

## 3. 발화 인덱스 + feature 빌드 (끊기거나 실패해도 `--resume` 붙여서 그대로 재실행하면 이어서 진행)

In [ ]:
index_output = str(CACHE_DIR / "m2_utterances_train_local_full.parquet")
features_output = str(CACHE_DIR / "m2_features_train_local_full.parquet")
failures_log = str(CACHE_DIR / "m2_build_failures_local_full.jsonl")

run_step(
    [
        "scripts/build_mission2_dataset.py",
        "--manifest", "data/manifests/dcc_manifest_local_full.csv",
        "--data-root", OUTPUT_DIR,
        "--n-calls", "0",
        "--seed", "42",
        "--index-output", index_output,
        "--features-output", features_output,
        "--failures-log", failures_log,
        "--workers", str(WORKERS),
        "--flush-every", "2000",  # 병렬이라 빨리 끝나므로 200마다 전체를 다시 쓰는 오버헤드를 줄임
        "--resume",
    ]
)

## 4. 학습 (기존 baseline과 동일한 GroupShuffleSplit(seed=42))

In [ ]:
model_output = str(CACHE_DIR / "m2_baseline_lgbm_local_full.pkl")
log_output = str(CACHE_DIR / "m2_exp_log_local_full.json")

run_step(
    [
        "scripts/train_mission2_baseline.py",
        "--features", features_output,
        "--model-output", model_output,
        "--log-output", log_output,
    ]
)

In [ ]:
import json
print(json.dumps(json.loads((CACHE_DIR / "m2_exp_log_local_full.json").read_text(encoding="utf-8")), ensure_ascii=False, indent=2))